In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [2]:
dataset1 = pd.read_csv("PreprocessedKidneyDisease.csv")
dataset1

,id,age,bp,sg,al,su,bgr,bu,sc,sod,...,pc,pcc,ba,htn,dm,cad,appet,pe,ane,classification
0,0.0,48.0,80.0,1.020,1.0,0.0,121.000000,36.0,1.2,137.528754,...,1,0,0,1,1,0,0,0,0,0
1,1.0,7.0,50.0,1.020,4.0,0.0,148.036517,18.0,0.8,137.528754,...,1,0,0,0,0,0,0,0,0,0
2,2.0,62.0,80.0,1.010,2.0,3.0,423.000000,53.0,1.8,137.528754,...,1,0,0,0,1,0,1,0,1,0
3,3.0,48.0,70.0,1.005,4.0,0.0,117.000000,56.0,3.8,111.000000,...,0,1,0,1,0,0,1,1,1,0
4,4.0,51.0,80.0,1.010,2.0,0.0,106.000000,26.0,1.4,137.528754,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395.0,55.0,80.0,1.020,0.0,0.0,140.000000,49.0,0.5,150.000000,...,1,0,0,0,0,0,0,0,0,1
396,396.0,42.0,70.0,1.025,0.0,0.0,75.000000,31.0,1.2,141.000000,...,1,0,0,0,0,0,0,0,0,1
397,397.0,12.0,80.0,1.020,0.0,0.0,100.000000,26.0,0.6,137.000000,...,1,0,0,0,0,0,0,0,0,1
398,398.0,17.0,60.0,1.025,0.0,0.0,114.000000,50.0,1.0,135.000000,...,1,0,0,0,0,0,0,0,0,1


In [3]:
print(dataset1.columns.tolist())

['id', 'age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


In [4]:

df2=dataset1
#df2 = pd.get_dummies(df2, drop_first=True)
indep_x = df2.drop(["classification", "id"], axis=1)
dep_y = df2["classification"]



In [5]:
def selectKBest(indep_x, dep_y, n):
    test = SelectKBest(chi2, k=n)
    fit1 = test.fit(indep_x, dep_y)
    selectk_features = fit1.transform(indep_x)
    selected_columns = indep_x.columns[fit1.get_support()]
    return selectk_features, selected_columns

def split_scalar(indep_x, dep_y):
    x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.25, random_state=0)
    sc = StandardScaler()
    x_train = sc.fit_transform(x_train)
    x_test = sc.transform(x_test)
    return x_train, x_test, y_train, y_test
    

In [6]:
Kbest, ColBest = selectKBest(indep_x, dep_y, 9)
x_train, x_test, y_train, y_test = split_scalar(Kbest, dep_y)


In [7]:
def cm_prediction(Classifier, x_test, y_test):
    from sklearn.metrics import confusion_matrix
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    y_pred = Classifier.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return Classifier, Accuracy, report, x_test, y_test, cm


In [8]:
#Logistic
def logistic(x_train, y_train, x_test, y_test):
    from sklearn.linear_model import LogisticRegression

    Classifier = LogisticRegression(random_state=0)

    Classifier.fit(x_train, y_train)

    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(
        Classifier, x_test, y_test
    )

    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMLinear
def svm_linear(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'linear', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMNonLinear
def svm_NL(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'rbf', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#Navie
def Navie(x_train, y_train, x_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    Classifier = GaussianNB()
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#KNN
def knn(x_train, y_train, x_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    Classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p=2)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#Decision
def Decision(x_train, y_train, x_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    Classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#RandomForest
def random(x_train, y_train, x_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    Classifier = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

In [9]:
def selectK_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf):
    dataframe = pd.DataFrame(index=['ChiSquare'], columns=['Logistic', 'SVML', 'SVMNL', 'KNN', 'Navie', 'Decision', 'Random'])

    for number, idex in enumerate(dataframe.index):
        dataframe.loc[idex, 'Logistic']=acclog[number]
        dataframe.loc[idex, 'SVML'] = accsvml[number]
        dataframe.loc[idex,'SVMNL'] = accsvmnl[number]
        dataframe.loc[idex, 'KNN'] = accknn[number]
        dataframe.loc[idex, 'Navie'] = accnav[number]
        dataframe.loc[idex, 'Decision'] = accdes[number]
        dataframe.loc[idex, 'Random'] = accrf[number]
    return dataframe

In [10]:
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []

Classifier, Accuracy, report, x_test, y_test, cm = logistic(x_train, y_train, x_test, y_test)
acclog.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = svm_linear(x_train, y_train, x_test, y_test)
accsvml.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = svm_NL(x_train, y_train, x_test, y_test)
accsvmnl.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = knn(x_train, y_train, x_test, y_test)
accknn.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = Navie(x_train, y_train, x_test, y_test)
accnav.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = Decision(x_train, y_train, x_test, y_test)
accdes.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
accrf.append(Accuracy)



In [11]:
result_df =selectK_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf)

# Comparison of each Model's Accuracy values

In [12]:
result_df

,Logistic,SVML,SVMNL,KNN,Navie,Decision,Random
ChiSquare,0.97,0.98,0.98,0.97,0.96,0.97,0.96


# Columns Selected for k=9

In [13]:
ColBest

Index(['age', 'al', 'su', 'bgr', 'bu', 'sc', 'hemo', 'pcv', 'wc'], dtype='object')